# Notebook 03 — Model Experiments

**Goal:** Test the Transformer architecture, simulate the full GNN → Transformer pipeline, and experiment with hyperparameters before committing to a full training run.

### Steps
1. Simulate a batch of GNN feature sequences
2. Run the Transformer forward pass and inspect outputs
3. Understand what the risk score and body region outputs mean
4. Try different Transformer sizes and compare parameter counts
5. Simulate a training step to verify gradients flow correctly

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from module1.transformer.model import BiomechanicalTransformer, BODY_REGIONS
from module1.gnn.model import SkeletonGAT
from utils.config import load_config

cfg = load_config('../configs/module1.yaml')
print('Config loaded OK')
print(f'Transformer config: {cfg["transformer"]}')

In [ ]:
# ── Simulate a batch of GNN output sequences ───────────────────────────────────
# In real use, these come from running the GNN on 50 consecutive frames.
# Here we generate random data to test the Transformer architecture.

BATCH_SIZE = 4
SEQ_LEN    = 50   # 2 seconds at 25fps
GNN_DIM    = 64

# 0 = healthy player, 1 = at-risk player
fake_sequences = torch.rand(BATCH_SIZE, SEQ_LEN, GNN_DIM)
fake_labels    = torch.tensor([0, 0, 1, 1])   # last 2 players are at-risk

print(f'Input sequence shape: {fake_sequences.shape}')  # (4, 50, 64)

In [ ]:
# ── Transformer forward pass ──────────────────────────────────────────────────
t_cfg = cfg['transformer']

model = BiomechanicalTransformer(
    input_dim  = cfg['gnn']['out_dim'],
    d_model    = t_cfg['d_model'],
    nhead      = t_cfg['nhead'],
    num_layers = t_cfg['num_layers'],
    dim_ff     = t_cfg['dim_feedforward'],
    dropout    = t_cfg['dropout'],
    seq_len    = t_cfg['seq_len'],
)
model.eval()

with torch.no_grad():
    out = model(fake_sequences)

print(f'Risk scores    : {out["risk_score"].squeeze().tolist()}')
print(f'Predicted regions: {[BODY_REGIONS[i] for i in out["region_logits"].argmax(dim=1).tolist()]}')
print(f'CLS embedding shape: {out["cls_embedding"].shape}')  # (4, 128)

In [ ]:
# ── Simulate one training step — verify gradients flow ────────────────────────
model.train()
optimiser  = torch.optim.Adam(model.parameters(), lr=0.001)
criterion  = nn.BCELoss()

optimiser.zero_grad()
out  = model(fake_sequences)
loss = criterion(out['risk_score'].squeeze().float(), fake_labels.float())
loss.backward()
optimiser.step()

print(f'Training loss (random weights, random data): {loss.item():.4f}')
print('Gradients flowed correctly — model is trainable.')

In [ ]:
# ── Compare model sizes ────────────────────────────────────────────────────────
def count_params(m): return sum(p.numel() for p in m.parameters())

configs_to_test = [
    {'d_model': 64,  'nhead': 2, 'num_layers': 1, 'dim_ff': 128,  'label': 'Tiny'},
    {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_ff': 256,  'label': 'Default'},
    {'d_model': 256, 'nhead': 8, 'num_layers': 4, 'dim_ff': 512,  'label': 'Large'},
]

print(f'{"Config":<12} {"Params":>12}  {"Output shape"}')
print('-' * 45)

for c in configs_to_test:
    m = BiomechanicalTransformer(
        input_dim=64,
        d_model=c['d_model'], nhead=c['nhead'],
        num_layers=c['num_layers'], dim_ff=c['dim_ff'],
        dropout=0.0, seq_len=50,
    )
    m.eval()
    with torch.no_grad():
        o = m(fake_sequences)
    print(f"{c['label']:<12} {count_params(m):>12,}  {tuple(o['risk_score'].shape)}")

In [ ]:
# ── Visualise attention: which frames does the model attend to? ───────────────
# (Advanced — only useful after the model is trained)
# For now, just print the CLS embedding norm across a sequence to get intuition.

model.eval()
with torch.no_grad():
    out = model(fake_sequences)

cls_norms = out['cls_embedding'].norm(dim=1).numpy()
plt.figure(figsize=(8, 3))
plt.bar(range(BATCH_SIZE), cls_norms, color=['steelblue', 'steelblue', 'tomato', 'tomato'])
plt.xticks(range(BATCH_SIZE), [f'P{i} ({"risk" if fake_labels[i] else "safe"})' for i in range(BATCH_SIZE)])
plt.ylabel('CLS embedding L2 norm')
plt.title('CLS token magnitude per player')
plt.show()